# 12 — VADER vs FinBERT: il finance-tuning paga? (backlog D1)

**Contesto**: ADR-023 ha scelto VADER (Layer 1, zero dipendenze) e ha subordinato l'upgrade a FinBERT all'**evidenza empirica**. Il notebook 06 (n=143) non trovò segnale lead col Layer 1. Oggi l'archivio ha ~7k titoli rilevanti per BTC/ETH: è il momento del confronto diretto.

**Setup**: 6.936 titoli (googlenews_btc/eth + Cointelegraph + CoinDesk) scorati con entrambi: VADER compound (già in archivio) e FinBERT (ProsusAI/finbert, P(pos)−P(neg), stesso range [−1,1]). Prezzi daily BTC/ETH dagli ultimi 365 giorni (serie committate). FinBERT gira come **tooling d'esperimento** (venv, non in pyproject): l'eventuale adozione passerebbe da un ADR.

## Ipotesi — scritte PRIMA di vedere i numeri

- **H1 (disaccordo materiale)**: VADER è general-domain: mi aspetto accordo di polarità **< 80%** sui titoli finanziari, con disaccordi sistematici su lessico di settore ("plunge", "outflows", "beats estimates", "hawkish").
- **H2 (allineamento same-day migliore)**: il sentiment giornaliero FinBERT dovrebbe correlare coi return **contemporanei** più di VADER (capisce il linguaggio finanziario, quindi descrive meglio la giornata).
- **H3 (nessun lead, per entrambi)**: coerente con nb 06 ed efficienza weak-form: sentiment(t) → return(t+1) ≈ 0 per entrambi gli scorer.

## Regola di decisione — pre-registrata
FinBERT entra in pipeline (nuovo ADR, dipendenze pesanti nel cron) **solo se**: la correlazione same-day migliora di **≥ +50% relativo su BTC ed ETH insieme**, *oppure* emerge un segnale lead assente in VADER. Altrimenti: si resta su VADER e si documenta (anche un no è un risultato).

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

SCORES = Path('../data/cache/finbert_scores.parquet')
assert SCORES.exists(), (
    'Cache mancante: uv pip install torch transformers, poi '
    'uv run --no-sync python notebooks/score_finbert_cache.py (dalla repo root).'
)
df = pd.read_parquet(SCORES)
ms = json.load(open('../public/data/market_series.json'))
px = {s['symbol']: pd.Series([p['v'] for p in s['points']],
      index=pd.DatetimeIndex(pd.to_datetime([p['t'] for p in s['points']]), tz='UTC'))
      for s in ms['series'] if s['symbol'] in ('BTC', 'ETH')}
ret = {k: v.pct_change().dropna() for k, v in px.items()}
print(f'{len(df)} titoli scorati | prezzi: '
      f"BTC {px['BTC'].index.min().date()}->{px['BTC'].index.max().date()}")
df[['vader', 'finbert']].describe().round(3)

## 1. H1 — Quanto (e dove) i due scorer sono in disaccordo
Accordo di polarità (soglia neutrale ±0.05), correlazione fra gli score, ed esempi dei disaccordi più estremi — per vedere *chi* dei due legge meglio i titoli finanziari.

In [ ]:
def pol(s, eps=0.05):
    return np.where(s > eps, 1, np.where(s < -eps, -1, 0))

pv, pf = pol(df['vader']), pol(df['finbert'])
agree = (pv == pf).mean()
corr_p = df['vader'].corr(df['finbert'])
corr_s = df['vader'].corr(df['finbert'], method='spearman')
print(f'accordo di polarità: {agree:.1%}   pearson={corr_p:.3f}   spearman={corr_s:.3f}')

opposite = df[(pv == 1) & (pf == -1) | (pv == -1) & (pf == 1)]
print(f'polarità OPPOSTE: {len(opposite)} titoli ({len(opposite)/len(df):.1%})')

d = (df['finbert'] - df['vader']).abs().sort_values(ascending=False)
print('\n--- disaccordi più estremi (titolo | vader | finbert) ---')
for i in d.index[:10]:
    r = df.loc[i]
    print(f"  {r['vader']:+.2f} | {r['finbert']:+.2f} | {r['title'][:90]}")

## 2. H2 — Allineamento coi return contemporanei
Sentiment medio giornaliero per asset (fonte dedicata + newswire) vs return dello stesso giorno UTC. Correlazione di Spearman (robusta agli outlier), su giorni con ≥ 3 titoli.

In [ ]:
ASSET_SOURCES = {'BTC': ['googlenews_btc', 'cointelegraph', 'coindesk'],
                 'ETH': ['googlenews_eth', 'cointelegraph', 'coindesk']}
MIN_TITLES = 3

def daily_sentiment(sym, col):
    sub = df[df['source'].isin(ASSET_SOURCES[sym])]
    day = pd.DatetimeIndex(sub.index).normalize()
    g = sub.groupby(day)[col].agg(['mean', 'count'])
    return g[g['count'] >= MIN_TITLES]['mean']

rows = []
for sym in ('BTC', 'ETH'):
    r = ret[sym]
    for col in ('vader', 'finbert'):
        s = daily_sentiment(sym, col)
        j = pd.concat([s, r], axis=1, join='inner').dropna()
        j.columns = ['sent', 'ret']
        rho, p = stats.spearmanr(j['sent'], j['ret'])
        rows.append({'asset': sym, 'scorer': col, 'giorni': len(j),
                     'spearman_same_day': round(rho, 3), 'p_value': round(p, 4)})
same_day = pd.DataFrame(rows).set_index(['asset', 'scorer'])
same_day

## 3. H3 — Lead (t → t+1) e reverse (il prezzo guida le news?)
`sent(t) vs ret(t+1)`: potere predittivo. `ret(t) vs sent(t+1)`: la stampa che insegue il prezzo (atteso dominante, da nb 06).

In [ ]:
rows = []
for sym in ('BTC', 'ETH'):
    r = ret[sym]
    for col in ('vader', 'finbert'):
        s = daily_sentiment(sym, col)
        j = pd.concat([s.rename('sent'), r.rename('ret')], axis=1).dropna()
        lead = stats.spearmanr(j['sent'].iloc[:-1], j['ret'].iloc[1:])
        rev = stats.spearmanr(j['ret'].iloc[:-1], j['sent'].iloc[1:])
        rows.append({'asset': sym, 'scorer': col,
                     'lead sent(t)->ret(t+1)': round(lead.statistic, 3),
                     'lead p': round(lead.pvalue, 3),
                     'reverse ret(t)->sent(t+1)': round(rev.statistic, 3),
                     'reverse p': round(rev.pvalue, 3)})
leads = pd.DataFrame(rows).set_index(['asset', 'scorer'])
leads

## 4. Verdetto

_(Da compilare coi numeri reali dopo l'esecuzione, contro la regola di decisione pre-registrata. Le ipotesi H1-H3 restano sopra, qualunque sia l'esito.)_